## Phase 5 — Full System Integration
## End-to-end single object tracking: Harris + LK on complete video
**Ali | 23L-2619 | BS Data Science 6th Semester | Fundamentals of Computer Vision |Final Project**

In [19]:
import sys, os, cv2, time, gc
import numpy as np

# ── Run everything from project root ──────────────────────────────────
PROJECT_ROOT = r"C:\Users\pakistan\OneDrive\Desktop\cv_project"
os.chdir(PROJECT_ROOT)
sys.path.insert(0, PROJECT_ROOT)

# ── Force reload all project modules fresh ────────────────────────────
for mod in list(sys.modules.keys()):
    if any(x in mod for x in ["core", "utils", "metrics", "visualization"]):
        del sys.modules[mod]

from utils.config_loader import ConfigLoader
from metrics.logger import AppLogger
from utils.folder_manager import FolderManager
from core.object_tracker import ObjectTracker
from visualization.display import DisplayRenderer

print("✅ All imports OK")

✅ All imports OK


In [20]:
cfg = ConfigLoader("config.yaml").config

print("=== SYSTEM CONFIG ===")
print(f"Video source       : {cfg.input.source}")
print(f"Harris max corners : {cfg.harris.max_corners}")
print(f"Harris quality     : {cfg.harris.quality_level}")
print(f"LK win size        : {cfg.lucas_kanade.win_size}")
print(f"FB error threshold : {cfg.lucas_kanade.fb_error_threshold}")
print(f"Min points         : {cfg.tracking.min_points_per_object}")
print(f"Redetect interval  : {cfg.tracking.redetect_interval}")
print(f"CLAHE enabled      : {cfg.preprocessing.clahe_enable}")
print()

# Verify video file exists
if not os.path.exists(cfg.input.source):
    raise FileNotFoundError(f"Video not found: {cfg.input.source}")

print("✅ Config verified — video file exists")

=== SYSTEM CONFIG ===
Video source       : C:\Users\pakistan\OneDrive\Desktop\cv_project\video\test_1.mp4
Harris max corners : 400
Harris quality     : 0.005
LK win size        : [41, 21]
FB error threshold : 3.5
Min points         : 4
Redetect interval  : 60
CLAHE enabled      : True

✅ Config verified — video file exists


In [21]:
import subprocess, sys, os

PROJECT_ROOT = r"C:\Users\pakistan\OneDrive\Desktop\cv_project"
MAIN_PY      = os.path.join(PROJECT_ROOT, "main.py")
CONFIG       = os.path.join(PROJECT_ROOT, "config.yaml")

print(f"Running : {MAIN_PY}")
print(f"Config  : {CONFIG}")
print("-" * 50)

result = subprocess.run(
    [sys.executable, MAIN_PY, "--config", CONFIG],
    capture_output=True,
    text=True,
    cwd=PROJECT_ROOT      # main.py opens config.yaml as a relative path internally
)

print(result.stdout)
if result.stderr:
    print("ERRORS:\n", result.stderr)

Running : C:\Users\pakistan\OneDrive\Desktop\cv_project\main.py
Config  : C:\Users\pakistan\OneDrive\Desktop\cv_project\config.yaml
--------------------------------------------------

ERRORS:
 [INFO] Starting Harris + LK single-object tracker.
[INFO] Video: 3840x2160 @ 24.0 fps
[INFO] Frame 0 selected as start frame.
[INFO] Selected bbox (full res): (600, 951, 444, 180)
[INFO] Output: results\test_1_20260516_173936\videos\test_1_tracked.mp4
[INFO] Frame 30 | Points: 64
[INFO] Frame 60 | Points: 48
[INFO] Frame 90 | Points: 37
[INFO] Frame 120 | Points: 37
[INFO] Frame 150 | Points: 37
[INFO] Frame 180 | Points: 37
[INFO] Frame 210 | Points: 36
[INFO] Frame 240 | Points: 36
[INFO] Frame 270 | Points: 36
[INFO] Done. Processed 280 frames.
[INFO] Output saved to: results\test_1_20260516_173936\videos\test_1_tracked.mp4

	OpenH264 Video Codec provided by Cisco Systems, Inc.




In [22]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os, sys, glob

PROJECT_ROOT = r"C:\Users\pakistan\OneDrive\Desktop\cv_project"
os.chdir(PROJECT_ROOT)

# Find the tracked video directly
matches = glob.glob(os.path.join(PROJECT_ROOT, "results", "**", "*_tracked.mp4"), recursive=True)

if not matches:
    raise FileNotFoundError("No tracked video found. Run tracking cell first.")

# Pick most recently modified
output_path = max(matches, key=os.path.getmtime)
input_name  = os.path.basename(output_path).replace("_tracked.mp4", "")
plots_dir   = os.path.join(os.path.dirname(output_path), "..", "plots")

print(f"Loading: {output_path}")

# Extract snapshots
cap   = cv2.VideoCapture(output_path)
total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

sample_indices = np.linspace(0, total-1, 6, dtype=int)
snapshots      = {}

for idx in sample_indices:
    cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
    ret, frame = cap.read()
    if ret:
        snapshots[idx] = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

cap.release()

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
axes = axes.flatten()

for ax, (fidx, snap) in zip(axes, snapshots.items()):
    ax.imshow(snap)
    ax.set_title(f"Frame {fidx}", fontsize=11)
    ax.axis("off")

plt.suptitle(f"Tracking Output — {input_name}", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(plots_dir, "integration_snapshots.png"), dpi=120)
plt.show()
print("✅ Snapshots saved")

Loading: C:\Users\pakistan\OneDrive\Desktop\cv_project\results\test_1_20260516_173936\videos\test_1_tracked.mp4
Figure(1600x800)
✅ Snapshots saved


In [23]:
import glob, os

PROJECT_ROOT = r"C:\Users\pakistan\OneDrive\Desktop\cv_project"

# Find most recent run.log
matches  = glob.glob(os.path.join(PROJECT_ROOT, "results", "**", "run.log"), recursive=True)

if not matches:
    raise FileNotFoundError("No log found. Run tracking cell first.")

log_path = max(matches, key=os.path.getmtime)
print(f"Log: {log_path}\n")

with open(log_path) as f:
    lines = f.readlines()

print(f"=== RUN LOG ({len(lines)} lines) ===\n")
for line in lines:
    print(line.strip())

Log: C:\Users\pakistan\OneDrive\Desktop\cv_project\results\test_1_20260516_173936\logs\run.log

=== RUN LOG (17 lines) ===

2026-05-16 17:39:36  [INFO    ]  Starting Harris + LK single-object tracker.
2026-05-16 17:39:37  [INFO    ]  Video: 3840x2160 @ 24.0 fps
2026-05-16 17:39:38  [INFO    ]  Frame 0 selected as start frame.
2026-05-16 17:39:38  [INFO    ]  Selected bbox (full res): (600, 951, 444, 180)
2026-05-16 17:39:39  [DEBUG   ]  [Tracker 1] Initialized with 67 points.
2026-05-16 17:39:40  [INFO    ]  Output: results\test_1_20260516_173936\videos\test_1_tracked.mp4
2026-05-16 17:40:01  [INFO    ]  Frame 30 | Points: 64
2026-05-16 17:40:19  [INFO    ]  Frame 60 | Points: 48
2026-05-16 17:40:36  [INFO    ]  Frame 90 | Points: 37
2026-05-16 17:40:54  [INFO    ]  Frame 120 | Points: 37
2026-05-16 17:41:12  [INFO    ]  Frame 150 | Points: 37
2026-05-16 17:41:30  [INFO    ]  Frame 180 | Points: 37
2026-05-16 17:41:48  [INFO    ]  Frame 210 | Points: 36
2026-05-16 17:42:06  [INFO    ] 

In [24]:
import json, glob, os, sys
import numpy as np
from datetime import datetime

PROJECT_ROOT = r"C:\Users\pakistan\OneDrive\Desktop\cv_project"
os.chdir(PROJECT_ROOT)
sys.path.insert(0, PROJECT_ROOT)

for mod in list(sys.modules.keys()):
    if any(x in mod for x in ["core","utils","metrics","visualization"]):
        del sys.modules[mod]

from utils.config_loader import ConfigLoader

cfg        = ConfigLoader("config.yaml").config
input_name = os.path.splitext(os.path.basename(cfg.input.source))[0]

# Find most recent results folder
matches     = glob.glob(os.path.join(PROJECT_ROOT, "results", "**", "*_tracked.mp4"), recursive=True)
output_path = max(matches, key=os.path.getmtime)
results_dir = os.path.dirname(os.path.dirname(output_path))
metrics_dir = os.path.join(results_dir, "metrics")
os.makedirs(metrics_dir, exist_ok=True)

# Parse log for point counts
log_path = max(
    glob.glob(os.path.join(PROJECT_ROOT, "results", "**", "run.log"), recursive=True),
    key=os.path.getmtime
)

point_counts = []
total_frames = 0
with open(log_path) as f:
    for line in f:
        if "Points:" in line and "Frame" in line:
            try:
                pts = int(line.strip().split("Points:")[-1].strip())
                point_counts.append(pts)
            except:
                pass
        if "Processed" in line:
            try:
                total_frames = int(line.strip().split("Processed")[-1].split("frames")[0].strip())
            except:
                pass

# Build and save metrics
metrics = {
    "test"        : input_name,
    "timestamp"   : datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "total_frames": total_frames,
    "avg_points"  : round(float(np.mean(point_counts)), 2) if point_counts else 0,
    "min_points"  : int(np.min(point_counts)) if point_counts else 0,
    "max_points"  : int(np.max(point_counts)) if point_counts else 0,
}

json_path = os.path.join(metrics_dir, f"{input_name}_metrics.json")
with open(json_path, "w") as f:
    json.dump(metrics, f, indent=4)

print(f"✅ Metrics saved: {json_path}")
print()
print("=== METRICS SUMMARY ===")
print(f"  Test             : {metrics['test']}")
print(f"  Timestamp        : {metrics['timestamp']}")
print(f"  Total frames     : {metrics['total_frames']}")
print(f"  Avg points/frame : {metrics['avg_points']}")
print(f"  Min points       : {metrics['min_points']}")
print(f"  Max points       : {metrics['max_points']}")

✅ Metrics saved: C:\Users\pakistan\OneDrive\Desktop\cv_project\results\test_1_20260516_173936\metrics\test_1_metrics.json

=== METRICS SUMMARY ===
  Test             : test_1
  Timestamp        : 2026-05-16 17:43:47
  Total frames     : 280
  Avg points/frame : 40.89
  Min points       : 36
  Max points       : 64


In [25]:
import json, glob, os, sys
from datetime import datetime

PROJECT_ROOT = r"C:\Users\pakistan\OneDrive\Desktop\cv_project"
os.chdir(PROJECT_ROOT)
sys.path.insert(0, PROJECT_ROOT)

for mod in list(sys.modules.keys()):
    if any(x in mod for x in ["core","utils","metrics","visualization"]):
        del sys.modules[mod]

from utils.config_loader import ConfigLoader

cfg        = ConfigLoader("config.yaml").config
input_name = os.path.splitext(os.path.basename(cfg.input.source))[0]

# Find most recent results folder
matches     = glob.glob(os.path.join(PROJECT_ROOT, "results", "**", "*_tracked.mp4"), recursive=True)
output_path = max(matches, key=os.path.getmtime)
results_dir = os.path.dirname(os.path.dirname(output_path))
params_dir  = os.path.join(results_dir, "params")
os.makedirs(params_dir, exist_ok=True)

# Build and save params
params = {
    "test"      : input_name,
    "timestamp" : datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "harris": {
        "max_corners"   : cfg.harris.max_corners,
        "quality_level" : cfg.harris.quality_level,
        "min_distance"  : cfg.harris.min_distance,
    },
    "lucas_kanade": {
        "win_size"          : cfg.lucas_kanade.win_size,
        "fb_error_threshold": cfg.lucas_kanade.fb_error_threshold,
    },
    "tracking": {
        "min_points_per_object": cfg.tracking.min_points_per_object,
        "redetect_interval"    : cfg.tracking.redetect_interval,
        "redetect_threshold"   : cfg.harris.redetect_threshold,
    },
    "preprocessing": {
        "clahe_enable": cfg.preprocessing.clahe_enable,
    }
}

params_path = os.path.join(params_dir, f"{input_name}_params.json")
with open(params_path, "w") as f:
    json.dump(params, f, indent=4)

print(f"✅ Params saved: {params_path}")
print()
print("=== PARAMS USED ===")
for section, values in params.items():
    if isinstance(values, dict):
        print(f"  [{section}]")
        for k, v in values.items():
            print(f"    {k:30s}: {v}")

✅ Params saved: C:\Users\pakistan\OneDrive\Desktop\cv_project\results\test_1_20260516_173936\params\test_1_params.json

=== PARAMS USED ===
  [harris]
    max_corners                   : 400
    quality_level                 : 0.005
    min_distance                  : 3
  [lucas_kanade]
    win_size                      : [41, 21]
    fb_error_threshold            : 3.5
  [tracking]
    min_points_per_object         : 4
    redetect_interval             : 60
    redetect_threshold            : 0.25
  [preprocessing]
    clahe_enable                  : True


In [27]:
# Aggregate all test metrics into one comparison file
all_metrics = []
all_json    = glob.glob(os.path.join(PROJECT_ROOT, "results", "**", "*_metrics.json"), recursive=True)

for path in sorted(all_json):
    with open(path) as f:
        all_metrics.append(json.load(f))

if all_metrics:
    combined_path = os.path.join(PROJECT_ROOT, "results", "all_tests_comparison.json")
    with open(combined_path, "w") as f:
        json.dump(all_metrics, f, indent=4)

    print(f"✅ Combined metrics saved: {combined_path}")
    print()
    print(f"{'Test':<12} {'Frames':>8} {'Avg Pts':>10} {'Min Pts':>10} {'Max Pts':>10}")
    print("-" * 55)
    for m in all_metrics:
        print(f"{m['test']:<12} {m['total_frames']:>8} {m['avg_points']:>10} {m['min_points']:>10} {m['max_points']:>10}")

✅ Combined metrics saved: C:\Users\pakistan\OneDrive\Desktop\cv_project\results\all_tests_comparison.json

Test           Frames    Avg Pts    Min Pts    Max Pts
-------------------------------------------------------
test_1            280       42.0         37         63
test_2            529      137.0        137        137


In [ ]:
## Summary
#- Config loaded from config.yaml — video source, Harris, LK, and tracking parameters verified
#- User-defined bounding box and start frame read from config (set once in Phase 2)
#- Harris corners detected inside bbox on the start frame as tracking anchors
#- Lucas-Kanade optical flow tracked corners frame-by-frame across the full video
#- Forward-backward error filter removed unreliable tracks each frame
#- Stationary background points filtered using velocity-guided displacement threshold
#- Median displacement of surviving object points shifted the bounding box each frame
#- Velocity estimate maintained to predict bbox position when point count drops
#- Harris redetection triggered when point survival dropped below 25% of initial count
#- Live display window showed tracking progress in real time
#- Output video saved to results/<video_name>_<timestamp>/videos/ in H.264 MP4 format
#- Tested on: test_1.mp4, test_2.mp4